# Visual-Language Navigation for Legged Robots using RGB Images
## CS 691 Course Project 

In [11]:
!pip install numpy torch Pillow pandas matplotlib tqdm scipy scikit-image datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 12.7 MB/s  0:00:00a 0:00:0136m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 13.7 MB/s  0:00:005.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 11.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers]0m 3/4 [transformers]


### Imports 

In [ ]:
import os, json, pickle, functools, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.spatial import KDTree
from skimage.draw import line_aa
from skimage.draw import line as sk_line
from datasets import load_dataset
from huggingface_hub import InferenceClient
import base64
import io
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

# We won't need this if we do ONLY API calls
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch : {torch.__version__}")
print(f"device : {DEVICE}")

torch  : 2.11.0+cu130
device : cpu


### Testing Model with API call

In [ ]:

client = InferenceClient(
    model="meta-llama/Llama-4-Scout-17B-16E-Instruct",
    token="" # add ur token here
)

response = client.chat_completion(
    messages=[{"role": "user", "content": "Who is this chat"}]
)

print(response.choices[0].message.content)

I'm Llama, a model designed by Meta. I'm here to adapt to your conversational style, whether you need quick answers, deep dives into ideas or just want to vent, joke or brainstorm.


### Loading in dataset

In [ ]:
dataset = load_dataset("leggedrobotics/navitrace")

sample = dataset["validation"][0]

for k, v in sample.items():
    if k == "image":
        print(f"image : Image {v.size} mode={v.mode}")

    elif k == "segmentation_mask":
        mask = np.array(v)
        print(f"segmentation_mask : shape={mask.shape} dtype={mask.dtype} unique classes={len(np.unique(mask))}")
        
    elif k == "ground_truth":
        print(f" ground_truth : embodiments={list(v.keys())}")
        for emb, traces in v.items():
            if traces is not None:
                lengths = [len(t) for t in traces]
                print(f" [{emb}]: {len(traces)} trace(s), lengths={lengths}")
            else:
                print(f" [{emb}]: None")
    elif k == "context":
        print(f" context : '{v[:80]}'")
    else:
        print(f" {k:<22}: {v}")

# showing sample
# sample['image']

 sample_id             : 1dfcd2f0-d1db-4197-9ee8-2e573145e9aa
 task                  : Go to the garden
 embodiments           : ['Human', 'Legged Robot']
image : Image (1365, 1024) mode=RGB
segmentation_mask : shape=(1024, 1365) dtype=int64 unique classes=5
 ground_truth : embodiments=['Bicycle', 'Human', 'Legged Robot', 'Wheeled Robot']
 [Bicycle]: None
 [Human]: 1 trace(s), lengths=[8]
 [Legged Robot]: 1 trace(s), lengths=[8]
 [Wheeled Robot]: None
 category              : ['Semantic Terrain', 'Geometric Terrain', 'Visibility']
 context : 'Corner of a house; Garden is downstairs behind the house; bike parking area is i'
 metadata              : {'city': 'Zurich', 'country': 'Switzerland', 'lighting_conditions': 'Daylight', 'natural_structured': 'Structured', 'task_type': 'Goal', 'urban_rural': 'Urban', 'weather_conditions': 'Cloudy'}


## Data Preprocessing (same as LLaVA)

- pad images to square
- map traces into padded normalized coordinates
- resample traces to a fixed number of waypoints
- convert normalized trace predictions back to pixel coordinates


In [7]:
def pad_to_square(image: Image.Image):
    w, h = image.size
    max_s = max(w, h)
    padded = Image.new("RGB", (max_s, max_s), (0, 0, 0))
    x_off = (max_s - w) // 2
    y_off = (max_s - h) // 2
    padded.paste(image, (x_off, y_off))
    return padded, (x_off / max_s, y_off / max_s, w / max_s, h / max_s)

def resize_if_needed(image: Image.Image, max_side: int):
    w, h = image.size
    longest = max(w, h)
    if longest <= max_side:
        return image
    scale = max_side / longest
    return image.resize((int(round(w * scale)), int(round(h * scale))), Image.BICUBIC)

def adjust_trace(trace_px, orig_w, orig_h, xof, yof, sx, sy):
    t = np.array(trace_px, dtype=float)
    if t.ndim != 2 or t.shape[1] != 2:
        raise ValueError("Trace must have shape (num_points, 2)")
    t[:, 0] = (t[:, 0] / orig_w) * sx + xof
    t[:, 1] = (t[:, 1] / orig_h) * sy + yof
    return t.astype(np.float32)

def interpolate_trace(trace: np.ndarray, N: int) -> np.ndarray:
    if len(trace) == 0:
        return np.zeros((N, 2), dtype=np.float32)
    if len(trace) == 1:
        return np.tile(trace[0], (N, 1)).astype(np.float32)

    diffs = np.diff(trace, axis=0)
    dists = np.sqrt((diffs ** 2).sum(axis=1))
    cumlen = np.concatenate([[0], np.cumsum(dists)])
    total = cumlen[-1]

    if total == 0:
        return np.tile(trace[0], (N, 1)).astype(np.float32)

    t_old = cumlen / total
    t_new = np.linspace(0, 1, N)
    out = np.stack(
        [
            np.interp(t_new, t_old, trace[:, 0]),
            np.interp(t_new, t_old, trace[:, 1]),
        ],
        axis=1,
    )
    return out.astype(np.float32)

def pred_padded_to_pixel(pred_norm, orig_w, orig_h):
    pred_norm = np.array(pred_norm, dtype=float)
    max_s = max(orig_w, orig_h)
    x_off = (max_s - orig_w) / 2.0
    y_off = (max_s - orig_h) / 2.0

    px = pred_norm.copy()
    px[:, 0] = px[:, 0] * max_s - x_off
    px[:, 1] = px[:, 1] * max_s - y_off

    px[:, 0] = np.clip(px[:, 0], 0, orig_w - 1)
    px[:, 1] = np.clip(px[:, 1], 0, orig_h - 1)
    return px.astype(np.float32)

def choose_canonical_trace(sample, N):
    # Main step comment: for generative supervision we need a single target sequence.
    gt_root = sample.get("ground_truth", None)
    if not isinstance(gt_root, dict):
        raise ValueError("Sample has no usable ground_truth dictionary")

    gt_traces = gt_root.get(CFG["embodiment"])
    if gt_traces is None or len(gt_traces) == 0:
        raise ValueError("Sample has no GT trace for the requested embodiment")

    gt_px = np.array(gt_traces[0], dtype=float)
    img = sample["image"]
    orig_w, orig_h = img.size
    _, (xof, yof, sx, sy) = pad_to_square(img)
    gt_norm = adjust_trace(gt_px, orig_w, orig_h, xof, yof, sx, sy)
    return interpolate_trace(gt_norm, N)

def compact_trace_json(trace_norm: np.ndarray, decimals: int = 4) -> str:
    # Main step comment: serialize the target trace as compact JSON for supervised generation.
    rounded = [[round(float(x), decimals), round(float(y), decimals)] for x, y in trace_norm]
    payload = {
        "goal": rounded[-1],
        "trace": rounded,
    }
    return json.dumps(payload, separators=(",", ":"))

def extract_first_json_object(text: str):
    # Main step comment: recover the model output as JSON even when generation contains extra text.
    if text is None:
        return None
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                snippet = text[start:i + 1]
                try:
                    return json.loads(snippet)
                except Exception:
                    return None
    return None

def json_to_trace_norm(obj, N):
    # Main step comment: safely convert model JSON back to a fixed-length normalized trace.

    if not isinstance(obj, dict) or "trace" not in obj:
        return None

    raw_trace = obj["trace"]
    if not isinstance(raw_trace, (list, tuple)):
        return None

    points = []

    for item in raw_trace:
        # Standard [x, y]
        if isinstance(item, (list, tuple)) and len(item) == 2:
            a, b = item[0], item[1]

            # [x, y]
            if np.isscalar(a) and np.isscalar(b):
                try:
                    points.append([float(a), float(b)])
                    continue
                except (TypeError, ValueError):
                    pass

            # [[x, y], extra] or [extra, [x, y]] -> salvage the pair if present
            for candidate in (a, b):
                if isinstance(candidate, (list, tuple)) and len(candidate) == 2:
                    try:
                        points.append([float(candidate[0]), float(candidate[1])])
                        break
                    except (TypeError, ValueError):
                        pass
            else:
                pass
            continue

        # Nested singleton like [[x, y]]
        if (
            isinstance(item, (list, tuple))
            and len(item) == 1
            and isinstance(item[0], (list, tuple))
            and len(item[0]) == 2
        ):
            try:
                points.append([float(item[0][0]), float(item[0][1])])
                continue
            except (TypeError, ValueError):
                pass

        # Dict point {"x": ..., "y": ...}
        if isinstance(item, dict) and "x" in item and "y" in item:
            try:
                points.append([float(item["x"]), float(item["y"])])
                continue
            except (TypeError, ValueError):
                pass

        # Flat 2-number dict variants
        if isinstance(item, dict) and "point" in item:
            p = item["point"]
            if isinstance(p, (list, tuple)) and len(p) == 2:
                try:
                    points.append([float(p[0]), float(p[1])])
                    continue
                except (TypeError, ValueError):
                    pass

    if len(points) == 0:
        return None

    trace = np.asarray(points, dtype=np.float32)

    if trace.ndim != 2 or trace.shape[1] != 2:
        return None

    trace = np.clip(trace, 0.0, 1.0)

    if len(trace) == N:
        return trace.tolist()

    if len(trace) == 1:
        return np.repeat(trace, N, axis=0).tolist()

    old_idx = np.linspace(0.0, 1.0, len(trace))
    new_idx = np.linspace(0.0, 1.0, N)
    x_new = np.interp(new_idx, old_idx, trace[:, 0])
    y_new = np.interp(new_idx, old_idx, trace[:, 1])

    return np.stack([x_new, y_new], axis=1).tolist()


### Llama prompting using Huggingface API

In [ ]:
# helper
def image_to_base64(image: Image.Image):
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

# Zero-shot prompt
def zero_shot_prompt(image: Image.Image, task: str, context: str):
    img = image_to_base64(image)
    
    response = client.chat_completion(
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img}"}},
                {"type": "text", "text": (
                    f"Context: {context}\n"
                    f"Given this outdoor scene and the instruction: {task}, "
                    "predict a navigation path for a legged robot as a sequence of "
                    "(x,y) waypoints from start to goal."
                )}
            ]
        }]
    )
    return response.choices[0].message.content


# Chain-of-thought prompt
def chain_of_thought_prompt(image: Image.Image, task: str, context: str):
    img = image_to_base64(image)

    steps = [
        f"Context: {context}\n",
        "Step 1: Describe the terrain type.",
        "Step 2: Identify the goal region described by the instruction.",
        "Step 3: Identify the obstacles or unsafe terrain to avoid.",
        f"Step 4: Identify the goal point as (x,y) and provide the navigation trace from start to goal for the instruction: {task}."
    ]
    cot_text = "\n".join(steps)

    response = client.chat_completion(
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img}"}},
                {"type": "text", "text": cot_text}
            ]
        }]
    )
    return response.choices[0].message.content

In [16]:
# testing for any results
results = []

for sample in tqdm(dataset["validation"].select(range(3))):  # only 3 now due to the limited api calls
    image   = sample["image"]
    task    = sample["task"]
    context = sample["context"]

    zs_output  = zero_shot_prompt(image, task, context)
    cot_output = chain_of_thought_prompt(image, task, context)

    results.append({
        "sample_id"  : sample["sample_id"],
        "task"       : task,
        "zero_shot"  : zs_output,
        "cot"        : cot_output,
    })

# print(results)

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

  0%|          | 0/3 [00:01<?, ?it/s]


HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-69ea5ad9-6efc769b02d3d65b6c5d542c;d8c9d9ee-9e34-44ad-88bc-1c891d2f2b68)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

### Loading in llama model instead of API Call

In [17]:
CFG = {
    "embodiment" : "Legged Robot",
    "model_id" : "meta-llama/Llama-4-Scout-17B-16E-Instruct",
    "max_new_tokens" : 512,
    "temperature" : 0.6,
    "val_split" : 0.2,
    "seed" : 42,
    "penalty_tsv" : "./category_penalty.tsv",
    "m2f_config" : "./mask2former_config.json",
    "bad_score_threshold" : 3234.75,
    "penalty_dist_thr" : 35,
    "N" : 10,
}

random.seed(CFG["seed"]),
np.random.seed(CFG["seed"])

In [ ]:
os.environ["HF_TOKEN"] = "" # put ur API key here

In [ ]:
quant_config = BitsAndBytesConfig(load_in_4bit=True)

processor = AutoProcessor.from_pretrained(CFG["model_id"])

llama_model = AutoModelForImageTextToText.from_pretrained(
    CFG["model_id"],
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.bfloat16
)
llama_model.eval()
print("llama 4 loaded boi :)")